# 04 — Lexical retrieval, dense baseline, and RRF fusion

## What this notebook proves

A question is mapped to a business profile and decomposed into required evidence fields. Each field receives explicit reformulations. BM25 retrieves exact labels/codes; Gemini retrieves learned semantic proximity; Reciprocal Rank Fusion (RRF) combines ranks without treating BM25 and cosine as comparable raw scales.

## Reading guide, cell by cell

1. **Bootstrap** reloads local modules so the kernel cannot retain obsolete code or corpus metadata.
2. **Question → profile → queries** prints nine queries: three fields × three reformulations. Mappings come from the evidence profile and QRT coordinates, not an LLM guess.
3. **Hybrid retrieval** loads the selected QRT chunks and prints full excerpts, chunk types, BM25 scores/ranks, Gemini cosine scores/ranks and fused RRF scores/ranks. IDs identify chunks; `excerpt` contains the evidence.
4. **Strategy comparison** measures field recall@k for a global question, one query per field and all reformulations. Recall 1.0 means every expected field has at least one relevant candidate in top-k; it does not prove answer correctness.
5. **Interpretation** states why a global query can miss structured QRT coordinates.
6. **Full corpus view** exposes every selected chunk to audit extraction, context duplication and visual/table descriptions.
7. **Stress cases** compare natural wording, regulatory labels and exact coordinates to expose BM25/Gemini disagreements.

## Stored-run conclusion

The global-question strategy recovers no validated field here; field-specific strategies recover all three. This supports the planner. Dense similarity and evidence validation are not contradictory: retrieval proposes likely text, while the contract checks field, entity, period and source. Retrieval proposes; the gate decides.

## Added value over classic RAG

A classic RAG usually sends one question to one top-k search and asks the generator to compose an answer. This project adds: (1) inspectable intent mapping, (2) an explicit list of required business fields, (3) multiple queries per field including regulatory coordinates, (4) lexical+dense rank fusion, (5) code-level entity/period/scope validation, (6) conflict and missing-evidence states, and (7) claim-level citations. The benefit is not merely a nicer answer: it reduces silent omissions and makes false completeness measurable.

The costs are equally explicit: profiles and mappings require domain maintenance; more queries increase latency and API usage; fixed triggers do not generalize to every wording; and a new business question may correctly fall into `open_question` until a reviewed contract is added. At larger volume, mappings should be versioned, evaluated on paraphrase sets and monitored for drift; retrieval needs batching and a persistent vector database.

In [31]:
from pathlib import Path
import sys

root_hint = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root_hint / 'notebooks'))
from helpers import bootstrap, display_table

ROOT = bootstrap()

for module_name in list(sys.modules):
    if module_name == 'app' or module_name.startswith('app.'):
        del sys.modules[module_name]

from app.domain.models import SearchQuery
from app.domain.profiles import explain_question_mapping, map_question
from app.retrieval.dense import GeminiDenseIndex
from app.retrieval.hybrid import HybridRetriever
from app.retrieval.planner import plan_queries
from ingestion.indexing import build_runtime_embedding_cache
from app.store.artifacts import store

In [32]:
question = "What public evidence describes Groupe Foyer's prudential coverage in 2025?"
document_ids = ['foyer_group_qrt_2025']
mapping_trace = explain_question_mapping(question)
display_table(mapping_trace)
profile = map_question(question)
print({'selected_profile': profile.id, 'required_fields': [field.id for field in profile.fields]})
queries = plan_queries(question, profile, document_ids)
display_table([
    {'field': query.field_id, 'reformulation': query.text}
    for query in queries
])

{'selected_profile': 'prudential_coverage', 'required_fields': ['eligible_own_funds_scr', 'group_scr', 'scr_coverage_ratio']}


,field,reformulation
0,eligible_own_funds_scr,S.23.01.22 R0660 C0010 eligible own funds covering total group SCR
1,eligible_own_funds_scr,total eligible own funds available to cover the group solvency capital requirement
2,eligible_own_funds_scr,Groupe Foyer prudential resources eligible for SCR coverage in 2025
3,group_scr,S.23.01.22 R0680 C0010 total group Solvency Capital Requirement
4,group_scr,Groupe Foyer total SCR amount in 2025
5,group_scr,consolidated group prudential capital requirement
6,scr_coverage_ratio,S.23.01.22 R0690 C0010 eligible own funds to total group SCR ratio
7,scr_coverage_ratio,Groupe Foyer Solvency Capital Requirement coverage rate
8,scr_coverage_ratio,consolidated SCR prudential coverage ratio in 2025


In [33]:
chunks = store.selected_chunks(document_ids)
index_directory = ROOT / 'data' / 'processed' / 'runtime_gemini_index'
manifest_path = index_directory / 'embedding_manifest.json'
if not manifest_path.exists():
    build_runtime_embedding_cache(chunks, index_directory, provider='gemini')
dense_index = GeminiDenseIndex(chunks, index_directory)
retriever = HybridRetriever(chunks, dense_index=dense_index)
print({'dense_provider': dense_index.provider, 'model': dense_index.model, 'chunks': len(chunks)})
rows = []
for query in queries:
    for candidate in retriever.search(query, k=3):
        rows.append({
            'field': query.field_id,
            'chunk_id': candidate.chunk.id,
            'chunk_type': candidate.chunk.chunk_type,
            'excerpt': candidate.chunk.text,
            'chunk_facts': ', '.join(fact.field_id for fact in candidate.chunk.facts),
            'source': candidate.chunk.locator.document_title,
            'page': candidate.chunk.locator.page,
            'lexical_rank': candidate.score.lexical_rank,
            'dense_rank': candidate.score.dense_rank,
            'score_rrf': candidate.score.rrf_score,
            'expected_fact': any(fact.field_id == query.field_id for fact in candidate.chunk.facts),
        })
display_table(rows)

{'dense_provider': 'gemini', 'model': 'gemini-embedding-001', 'chunks': 87}


,field,chunk_id,chunk_type,excerpt,chunk_facts,source,page,lexical_rank,dense_rank,score_rrf,expected_fact
0,eligible_own_funds_scr,foyer_group_qrt_2025-s230122-r0660-c0010,verified_table_evidence,S.23.01.22 — R0660/C0010: Total des fonds propres éligibles pour couvrir le SCR total du groupe: 2.407.647.,eligible_own_funds_scr,QRT public 2025 - Groupe Foyer,7,2,1,0.032522,True
1,eligible_own_funds_scr,foyer_group_qrt_2025-s230122-r0690-c0010,verified_table_evidence,"S.23.01.22 — R0690/C0010: Ratio total des fonds propres éligibles sur SCR total du groupe: 2,87.",scr_coverage_ratio,QRT public 2025 - Groupe Foyer,7,3,2,0.032002,False
2,eligible_own_funds_scr,foyer_group_qrt_2025-figure-004-visual-description-00,table_description,"Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Section: Groupe Foyer. Content: Visual type: table. Factual description: The image displays a table titled 'Fonds propres' (Own Funds) from 'Groupe Foyer'. It presents data across five columns: 'Total', 'Niveau 1 - non restreint', 'Niveau 1 - restreint', 'Niveau 2', and 'Niveau 3', identified by codes C0010 to C0050. The table details various categories of own funds and solvency capital requirements, with specific values provided for 'Total' and 'Niveau 1 - non restreint' columns. Key figures include total own funds from other financial sectors (R0440) at 80,957 for both 'Total' and 'Niveau 1 - non restreint', and total available own funds to cover the consolidated group SCR (R0520) at 2,326,690 for both categories. The minimum consolidated solvency capital requirement (R0610) is 308,997. The ratio of eligible own funds to the minimum capital requirement (R0650) is 7.53, and the ratio of total eligible own funds to total group SCR (R0690) is 2.87. Declared uncertainties: The specific currency or unit for the numerical values (e.g., thousands, millions) is not explicitly stated in the table.. Directly visible observation: Établissements de crédit, entreprises d'investissement, établissements financiers, gestionnaires de fonds d'investissement alternatifs (R0410) - Total (C0010) = 61554.0. Directly visible observation: Établissements de crédit, entreprises d'investissement, établissements financiers, gestionnaires de fonds d'investissement alternatifs (R0410) - Niveau 1 - non restreint (C0020) = 61554.0. Directly visible observation: Institution de retraite professionnelle (R0420) - Total (C0010) = 0.0. Directly visible observation: Institution de retraite professionnelle (R0420) - Niveau 1 - non restreint (C0020) = 0.0.",,QRT public 2025 - Groupe Foyer,7,1,7,0.031319,False
3,eligible_own_funds_scr,foyer_group_qrt_2025-figure-004-visual-description-00,table_description,"Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Section: Groupe Foyer. Content: Visual type: table. Factual description: The image displays a table titled 'Fonds propres' (Own Funds) from 'Groupe Foyer'. It presents data across five columns: 'Total', 'Niveau 1 - non restreint', 'Niveau 1 - restreint', 'Niveau 2', and 'Niveau 3', identified by codes C0010 to C0050. The table details various categories of own funds and solvency capital requirements, with specific values provided for 'Total' and 'Niveau 1 - non restreint' columns. Key figures include total own funds from other financial sectors (R0440) at 80,957 for both 'Total' and 'Niveau 1 - non restreint', and total available own funds to cover the consolidated group SCR (R0520) at 2,326,690 for both categories. The minimum consolidated solvency capital requirement (R0610) is 308,997. The ratio of eligible own funds to the minimum capital requirement (R0650) is 7.53, and the ratio of total eligible own funds to total group SCR (R0690) is 2.87. Declared uncertainties: The specific currency or unit for the numerical values (e.g., thousands, millions) is not explicitly stated in the table.. Directly visible observation: Établissements de crédit, entreprises d'investissement, établisseme

In [34]:
global_queries = [
    SearchQuery(
        field_id=field.id, field_label=field.label, text=question,
        document_ids=document_ids,
    )
    for field in profile.fields
]
first_by_field = {}
for query in queries:
    first_by_field.setdefault(query.field_id, query)
strategies = {
    'global_question': global_queries,
    'one_query_per_field': list(first_by_field.values()),
    'field_reformulations': queries,
}
recall_rows = []
required_fields = {field.id for field in profile.fields}
for strategy, strategy_queries in strategies.items():
    for k in (1, 3, 5, 10):
        recovered_fields = {
            query.field_id
            for query in strategy_queries
            if any(
                fact.field_id == query.field_id
                for candidate in retriever.search(query, k=k)
                for fact in candidate.chunk.facts
            )
        }
        table_context_fields = {
            query.field_id
            for query in strategy_queries
            if any(
                candidate.chunk.chunk_type == 'table_description'
                for candidate in retriever.search(query, k=k)
            )
        }
        recall_rows.append({
            'strategy': strategy,
            'k': k,
            'recovered_fields': sorted(recovered_fields),
            'verified_evidence_recall': len(recovered_fields) / len(required_fields),
            'table_context_recall': len(table_context_fields) / len(required_fields),
        })
assert any(
    row['strategy'] == 'field_reformulations'
    and row['verified_evidence_recall'] == 1.0
    for row in recall_rows
)
display_table(recall_rows)
classic_at_10 = next(row for row in recall_rows if row['strategy'] == 'global_question' and row['k'] == 10)
contract_at_10 = next(row for row in recall_rows if row['strategy'] == 'field_reformulations' and row['k'] == 10)
print({
    'comparison': 'classic single-query RAG vs field-contract retrieval',
    'classic_verified_recall_at_10': classic_at_10['verified_evidence_recall'],
    'contract_verified_recall_at_10': contract_at_10['verified_evidence_recall'],
    'measured_gain': contract_at_10['verified_evidence_recall'] - classic_at_10['verified_evidence_recall'],
    'interpretation': 'Gain on this QRT fixture only; not a universal benchmark.',
})

{'comparison': 'classic single-query RAG vs field-contract retrieval', 'classic_verified_recall_at_10': 0.0, 'contract_verified_recall_at_10': 1.0, 'measured_gain': 1.0, 'interpretation': 'Gain on this QRT fixture only; not a universal benchmark.'}


## Exhaustive corpus and query-form stress test

The following cells deliberately disable dataframe truncation. They expose every selected chunk and every rank for several question forms. This is diagnostic output, not the compact production UI.

In [35]:
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
corpus_rows = [
    {
        'chunk_id': chunk.id,
        'document_id': chunk.document_id,
        'chunk_type': chunk.chunk_type,
        'page': chunk.locator.page,
        'section': ' > '.join(chunk.locator.section_path),
        'characters': len(chunk.text),
        'fact_fields': ', '.join(fact.field_id for fact in chunk.facts),
        'full_text': chunk.text,
    }
    for chunk in chunks
]
assert len(corpus_rows) == len(chunks)
display_table(corpus_rows)

,chunk_id,document_id,chunk_type,page,section,characters,fact_fields,full_text
0,foyer_group_qrt_2025-docling-0000-00,foyer_group_qrt_2025,docling_hierarchical_text,1,Groupe Foyer,125,,Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Section: Groupe Foyer. Content: QRT Public 2025
1,foyer_group_qrt_2025-docling-0001-00,foyer_group_qrt_2025,docling_hierarchical_text,1,S.02.01.02,113,,Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Section: S.02.01.02. Content: Bilan
2,foyer_group_qrt_2025-docling-0002-00,foyer_group_qrt_2025,docling_hierarchical_text,1,S.02.01.02,1908,,"Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Section: S.02.01.02. Content: Actifs, Valeur Solvabilité II = . Actifs, = . Immobilisations incorporelles, Valeur Solvabilité II = R0030. Immobilisations incorporelles, = . Actifs d'impôts différés, Valeur Solvabilité II = R0040. Actifs d'impôts différés, = . Excédent du régime de retraite, Valeur Solvabilité II = R0050. Excédent du régime de retraite, = . Immobilisations corporelles détenues pour usage propre, Valeur Solvabilité II = R0060. Immobilisations corporelles détenues pour usage propre, = 119.859. Investissements (autres qu'actifs en représentation de contrats en unités de compte et indexés), Valeur Solvabilité II = R0070. Investissements (autres qu'actifs en représentation de contrats en unités de compte et indexés), = 4.241.339. Biens immobiliers (autres que détenus pour usage propre), Valeur Solvabilité II = R0080. Biens immobiliers (autres que détenus pour usage propre), = 108.248. Détentions dans des entreprises liées, y compris participations, Valeur Solvabilité II = R0090. Détentions dans des entreprises liées, y compris participations, = 108.114. Actions, Valeur Solvabilité II = R0100. Actions, = 608.154. Actions - cotées, Valeur Solvabilité II = R0110. Actions - cotées, = 583.250. Actions - non cotées, Valeur Solvabilité II = R0120. Actions - non cotées, = 24.904. Obligations, Valeur Solvabilité II = R0130. Obligations, = 2.713.529. Obligations d'État, Valeur Solvabilité II = R0140. Obligations d'État, = 949.952. Obligations d'entreprise, Valeur Solvabilité II = R0150. Obligations d'entreprise, = 1.232.926. Titres structurés, Valeur Solvabilité II = R0160. Titres structurés, = 530.651. Titres garantis, Valeur Solvabilité II = R0170. Titres garantis, = 0. Organismes de placement collectif, Valeur Solvabilité II = R0180. Organismes de placement collectif, = 640.456."
3,foyer_group_qrt_2025-docling-0002-01,foyer_group_qrt_2025,docling_hierarchical_text,1,S.02.01.02,1907,,"Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Section: S.02.01.02. Content: Produits dérivés, Valeur Solvabilité II = R0190. Produits dérivés, = 10.985. Dépôts autres que les équivalents de trésorerie, Valeur Solvabilité II = R0200. Dépôts autres que les équivalents de trésorerie, = 21.397. Autres investissements, Valeur Solvabilité II = R0210. Autres investissements, = 30.455. Actifs en représentation de contrats en unités de compte et indexés, Valeur Solvabilité II = R0220. Actifs en représentation de contrats en unités de compte et indexés, = 20.948.552. Prêts et prêts hypothécaires, Valeur Solvabilité II = R0230. Prêts et prêts hypothécaires, = 48.935. Avances sur police, Valeur Solvabilité II = R0240. Avances sur police, = 631. Prêts et prêts hypothécaires aux particuliers, Valeur Solvabilité II = R0250. Prêts et prêts hypothécaires aux particuliers, = 17.432. Autres prêts et prêts hypothécaires, Valeur Solvabilité II = R0260. Autres prêts et prêts hypothécaires, = 30.872. Montants recouvrables au titre des contrats de réassurance, Valeur Solvabilité II = R0270. Montants recouvrables au titre des contrats de réassurance, = 61.719. Non-vie et santé similaire à la non-vie, Valeur Solvabilité II = R0280. Non-vie et santé similaire à la non-vie, = 60.293. Non-vie hors santé, Valeur Solvabilité II = R0290. Non-vi

In [36]:
stress_cases = [
    ('natural_question', "What public evidence describes Groupe Foyer's prudential coverage in 2025?"),
    ('exact_regulatory_terms', 'Eligible own funds, group SCR and SCR coverage ratio for Groupe Foyer at year-end 2025'),
    ('acronym_heavy', 'Foyer Group EOF SCR ratio S.23.01.22 R0660 R0680 R0690 C0010 2025'),
    ('short_keyword_query', 'Foyer solvency 2025'),
    ('wrong_period', 'What was Groupe Foyer SCR coverage in 2024?'),
    ('wrong_entity', 'What was Foyer Assurances SCR coverage in 2025?'),
]
ranking_rows = []
for case_id, query_text in stress_cases:
    diagnostic_query = SearchQuery(
        field_id='diagnostic',
        field_label='Diagnostic query',
        text=query_text,
        document_ids=document_ids,
    )
    for final_rank, candidate in enumerate(
        retriever.search(diagnostic_query, k=len(chunks)), start=1
    ):
        ranking_rows.append({
            'case': case_id,
            'query': query_text,
            'final_rank': final_rank,
            'chunk_id': candidate.chunk.id,
            'chunk_type': candidate.chunk.chunk_type,
            'page': candidate.chunk.locator.page,
            'bm25_rank': candidate.score.lexical_rank,
            'bm25_score': candidate.score.lexical_score,
            'gemini_dense_rank': candidate.score.dense_rank,
            'gemini_cosine_score': candidate.score.dense_score,
            'rrf_score': candidate.score.rrf_score,
            'fact_fields': ', '.join(fact.field_id for fact in candidate.chunk.facts),
            'full_text': candidate.chunk.text,
        })
assert len(ranking_rows) == len(stress_cases) * len(chunks)
display_table(ranking_rows)

,case,query,final_rank,chunk_id,chunk_type,page,bm25_rank,bm25_score,gemini_dense_rank,gemini_cosine_score,rrf_score,fact_fields,full_text
0,natural_question,What public evidence describes Groupe Foyer's prudential coverage in 2025?,1,foyer_group_qrt_2025-docling-0021-00,docling_hierarchical_text,9,12,0.735998,3,0.799933,0.029762,,Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Section: S.25.01.22. Content: Capital de solvabilité requis - pour les groupes qui utilisent la formule standard
1,natural_question,What public evidence describes Groupe Foyer's prudential coverage in 2025?,2,foyer_group_qrt_2025-docling-0010-00,docling_hierarchical_text,4,4,0.755757,16,0.795342,0.028783,,Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Section: Groupe Foyer. Content: S.05.01.02 - 02
2,natural_question,What public evidence describes Groupe Foyer's prudential coverage in 2025?,3,foyer_group_qrt_2025-figure-006-visual-description-02,table_description,10,2,0.854877,23,0.782062,0.028177,,"Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Section: Groupe Foyer. Content: Directly visible observation: Code d'identification de l'entreprise (C0020) for Foyer Assurances S.A. = unreadable value. Directly visible observation: Nom juridique de l'entreprise (C0040) = unreadable value. Directly visible observation: % de part de capital (C0180) for Foyer Assurances S.A. = 1.0 %. Directly visible observation: Méthode utilisée et, en cas d'utilisation de la première méthode, traitement de l'entreprise (C0260) for Foyer Assurances S.A. = 1.0. Directly visible observation: Code d'identification de l'entreprise (C0020) for Foyer Vie S.A. = unreadable value. Directly visible observation: Nom juridique de l'entreprise (C0040) = unreadable value. Directly visible observation: Catégorie (C0070) for Foyer Vie S.A. = 2.0. Directly visible observation: % utilisé pour l'établissement des comptes consolidés (C0190) for Foyer Vie S.A. = 1.0 %. Directly visible observation: Autorité de contrôle (C0080) for Foyer-Arag S.A. = unreadable value. Directly visible observation: Part proportionnelle utilisée pour le calcul de la solvabilité du groupe (C0230) for CapitalatWork Foyer Group S.A. = 1.0. Directly visible observation: Type d'entreprise (C0050) for Raiffeisen Vie S.A. = 1.0. Directly visible observation: % de part de capital (C0180) for Foyer Global Health S.A. = 1.0 %. Directly visible observation: Autorité de contrôle (C0080) for Avise S.A. = unreadable value. Directly visible observation: % des droits de vote (C0200) for NexFin S.A. = 1.0 %. Directly visible observation: Méthode utilisée et, en cas d'utilisation de la première méthode, traitement de l'entreprise (C0260) for NexFin S.A. = 3.0."
3,natural_question,What public evidence describes Groupe Foyer's prudential coverage in 2025?,4,foyer_group_qrt_2025-figure-005-visual-description-02,table_description,9,13,0.695042,15,0.796140,0.027032,,Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Section: S.25.01.22. Content: Directly visible observation: SCR pour les entreprises incluses par la méthode D&A (R0560) = 0.0. Directly visible observation: Capital de solvabilité requis du groupe (R0570) = 840.04.
4,natural_question,What public evidence describes Groupe Foyer's prudential coverage in 2025?,5,foyer_group_qrt_2025-docling-0018-00,docling_hierarchical_text,8,9,0.749344,21,0.785941,0.026838,,Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Section: Groupe Foyer. Content: S.23.01.22 - 02 Fonds propres
5,natural_question,What public evidence describes Groupe Foyer's prudential coverage in 2025?,6,foyer_group_qrt_2025-docling-0001-00,docling_hierarchical_text,1,5,0.755609,29,0.779712,0.026621,,Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Section: S.02.01.02. Content: Bilan
6,natural_question,What public evidence describes Groupe Foyer's pr